In [12]:
import pandas as pd

df = pd.read_stata("/UgandaLSMS/Wave_1/AGSEC10.dta", convert_categoricals=False)

df.columns = df.columns.str.strip()

df["Hhid"] = df["Hhid"].apply(
    lambda x: f"{x:.0f}" if pd.notna(x) and isinstance(x, (int, float)) else str(x).strip()
)

df["A10q3"] = pd.to_numeric(df["A10q3"], errors="coerce")

df["any_A10q3_1"] = (df["A10q3"] == 1).astype(int)

out = df.groupby("Hhid", as_index=False)["any_A10q3_1"].max()

out["Hhid"] = "'" + out["Hhid"]

out.to_excel(r"C:\Users\Carl\Desktop\hhidextensiondummy.xlsx", index=False)

In [1]:
#loading module

import pandas as pd
from pathlib import Path


ROOT = Path("/UgandaLSMS/Wave_1")


def load_dta(path):
    df = pd.read_stata(path, convert_categoricals=False)
    df.columns = df.columns.str.strip().str.upper()
    return df


def check_keys(df, keys, name):
    print(f"\n{name}")
    print(f"rows: {len(df):,}")
    print(f"unique {keys}: {df[list(keys)].drop_duplicates().shape[0]:,}")
    dup = df.duplicated(subset=list(keys)).sum()
    print(f"duplicates on {keys}: {dup:,}")


In [4]:
# file wd hh

gsec1 = load_dta(ROOT / "GSEC1.dta")
gsec2 = load_dta(ROOT / "GSEC2.dta")
gsec3 = load_dta(ROOT / "GSEC3.dta")
gsec4 = load_dta(ROOT / "GSEC4.dta")
gsec9 = load_dta(ROOT / "GSEC9.dta")
gsec10 = load_dta(ROOT / "GSEC10.dta")
gsec10a = load_dta(ROOT / "GSEC10A.dta")
gsec11 = load_dta(ROOT / "GSEC11.dta")
gsec12 = load_dta(ROOT / "GSEC12.dta")
gsec13 = load_dta(ROOT / "GSEC13.dta")
gsec14 = load_dta(ROOT / "GSEC14.dta")
gsec15a = load_dta(ROOT / "GSEC15A.dta")
gsec15b = load_dta(ROOT / "GSEC15B.dta")
gsec15c = load_dta(ROOT / "GSEC15C.dta")
gsec15d = load_dta(ROOT / "GSEC15D.dta")
gsec15e = load_dta(ROOT / "GSEC15E.dta")
gsec16 = load_dta(ROOT / "GSEC16.dta")
gsec17 = load_dta(ROOT / "GSEC17.dta")
gsec18 = load_dta(ROOT / "GSEC18.dta")
gsec18a = load_dta(ROOT / "GSEC18A.dta")
gsec18b = load_dta(ROOT / "GSEC18B.dta")

geovars = load_dta(ROOT / "UNPS_Geovars_0910.dta")
pov09 = load_dta(ROOT / "pov2009_10.dta")

In [5]:
def keep_existing(df, cols):

    cols_upper = [c.upper() for c in cols]
    existing = [c for c in cols_upper if c in df.columns]
    return df[existing].copy()


def standardize_hhid(df):

    df = df.copy()
    df.columns = df.columns.str.strip().str.upper()

    if "HHID" not in df.columns:
        raise KeyError("HHID not found in dataframe")

    df["HHID"] = df["HHID"].apply(lambda x: f"{x:.0f}" if pd.notna(x) and isinstance(x, (int, float)) else str(x).strip())
    return df


def rename_nonkey(df, prefix):

    df = df.copy()
    rename_map = {
        c: f"{prefix}__{c}"
        for c in df.columns
        if c != "HHID"
    }
    return df.rename(columns=rename_map)


def prep_section(df, keep_cols, prefix):

    df = df.copy()
    df.columns = df.columns.str.strip().str.upper()

    df = keep_existing(df, keep_cols)

    df = standardize_hhid(df)

    df = df.drop_duplicates(subset=["HHID"])

    return df



In [6]:
def keep_existing(df, cols):

    cols_upper = [c.upper() for c in cols]
    existing = [c for c in cols_upper if c in df.columns]
    return df[existing].copy()


def standardize_pid_keys(df):

    df = df.copy()
    df.columns = df.columns.str.strip().str.upper()

    if "HHID" not in df.columns:
        raise KeyError("HHID not found in dataframe")
    if "PID" not in df.columns:
        raise KeyError("PID not found in dataframe")

    df["HHID"] = df["HHID"].apply(lambda x: f"{x:.0f}" if pd.notna(x) and isinstance(x, (int, float)) else str(x).strip())
    df["PID"] = df["PID"].astype(str).str.strip()

    return df


def prep_pid_section(df, keep_cols):

    df = df.copy()
    df.columns = df.columns.str.strip().str.upper()

    df = keep_existing(df, keep_cols)
    df = standardize_pid_keys(df)

    dup_count = df.duplicated(subset=["HHID", "PID"]).sum()
    print(f"Duplicates on HHID-PID: {dup_count}")

    if dup_count > 0:
        print("Warning: duplicates found. Inspect before merging.")
        dup = df[df.duplicated(subset=["HHID", "PID"], keep=False)].sort_values(["HHID", "PID"])
        print(dup.head(20))

    df = df.drop_duplicates(subset=["HHID", "PID"])

    return df


In [7]:
# 2. Variable list HHID

vars_gsec9 = [
    "HHID", "H9Q07", "H9Q08", "H9Q09A", "H9Q16", "H9Q20", "H9Q21", "H9Q22", "H9Q23"
]

vars_gsec10 = [
    "HHID", "H10Q13", "H10Q14", "H10Q17A"
]


vars_gsec10a = [
    "HHID", "H10Q1", "H10Q4", "H10Q6"
]

vars_gsec11 = [
    "HHID", "H11Q01", "H11AQ03", "H11AQ04", "H11AQ05", "H11AQ06", "H11AQ07"
]

vars_gsec12 = [
    "HHID", "H12Q01", "H12Q04", "H12Q09", "H12Q10", "H12Q13", "H12Q14",
    "H12Q18", "H12Q19"
]

vars_gsec13 = [
    "HHID", "H13Q01", "H13Q02", "H13Q03", "H13Q04", "H13Q05", "H13Q06",
    "H13Q07", "H13Q08", "H13Q09", "H13Q10", "H13Q11", "H13Q12", "H13Q13",
    "H13Q14", "H13Q15", "H13Q16", "H13Q17", "H13Q19", "H13Q20", "H13Q21",
    "H13Q22", "H13Q23", "H13Q24", "H13Q25"
]

vars_gsec14 = [
    "HHID", "H14Q2", "H14Q3", "H14Q4", "H14Q5"
]

vars_gsec15a = [
    "HHID", "H15A1", "H15A2", "H15A3", "H15A4", "H15A5", "H15A6", "H15A7", "H15A8"
]

vars_gsec15b = [
    "HHID", "H15BQ2", "H15BQ3B", "H15BQ4", "H15BQ5", "H15BQ6", "H15BQ7",
    "H15BQ8", "H15BQ9", "H15BQ10", "H15BQ11", "H15BQ12", "H15BQ13"
]

vars_gsec15c = [
    "HHID", "H15CQ2", "H15CQ3", "H15CQ4", "H15CQ5", "H15CQ6", "H15CQ7",
    "H15CQ8", "H15CQ9", "H15CQ10"
]

vars_gsec15d = [
    "HHID", "H15DQ3", "H15DQ4", "H15DQ5"
]

vars_gsec15e = [
    "HHID", "YEAR", "H15EQ2", "H15EQ3"
]

vars_gsec16 = [
    "HHID", "H16Q00", "H16Q01", "H16Q02B", "H16Q3A", "H16Q3B", "H16Q3C",
    "H16Q3D", "H16Q4A", "H16Q4B", "H16Q4C"
]

vars_gsec17 = [
    "HHID", "H17Q09", "H17Q10", "H17Q11"
]

vars_gsec18 = [
    "HHID", "H18Q1", "H18Q2", "H18Q5", "H18Q6"
]

vars_gsec18a = [
    "HHID", "H18Q7"
]

vars_gsec18b = [
    "HHID", "H18Q9", "H18Q10", "H18Q11"
]

vars_pov09 = [
    "HHID", "URBAN", "SPLITOFF", "REGION", "COMM", "DISTRICT", "REGURB",
    "EQUIV_M", "HSIZE_M", "NRREXP30", "CPEXP30", "WELFARE", "WELFARE1"
]

vars_unps = ["HHID", "LAT_MOD", "LON_MOD", "DIST_ROAD", "DIST_POPCENTER", "DIST_MARKET","FSRAD3_AGPCT", "FSRAD3_LCMAJ", "SSA_AEZ09", "AFMNSLP_PCT", "SRTM_UGA"]




sec_gsec9 = prep_section(gsec9, vars_gsec9, "GSEC9")
sec_gsec10 = prep_section(gsec10, vars_gsec10, "GSEC10")
sec_gsec10a = prep_section(gsec10, vars_gsec10a, "GSEC10A")
sec_gsec11 = prep_section(gsec11, vars_gsec11, "GSEC11")
sec_gsec12 = prep_section(gsec12, vars_gsec12, "GSEC12")
sec_gsec13 = prep_section(gsec13, vars_gsec13, "GSEC13")
sec_gsec14 = prep_section(gsec14, vars_gsec14, "GSEC14")
sec_gsec15a = prep_section(gsec15a, vars_gsec15a, "GSEC15A")
sec_gsec15b = prep_section(gsec15b, vars_gsec15b, "GSEC15B")
sec_gsec15c = prep_section(gsec15c, vars_gsec15c, "GSEC15C")
sec_gsec15d = prep_section(gsec15d, vars_gsec15d, "GSEC15D")
sec_gsec15e = prep_section(gsec15e, vars_gsec15e, "GSEC15E")
sec_gsec16 = prep_section(gsec16, vars_gsec16, "GSEC16")
sec_gsec17 = prep_section(gsec17, vars_gsec17, "GSEC17")
sec_gsec18 = prep_section(gsec18, vars_gsec18, "GSEC18")
sec_gsec18a = prep_section(gsec18a, vars_gsec18a, "GSEC18A")
sec_gsec18b = prep_section(gsec18b, vars_gsec18b, "GSEC18B")
sec_unps = prep_section(geovars, vars_unps, "UNPS_Geovars_0910")


hh_sections = [
    sec_gsec9,
    sec_gsec10,
    sec_gsec10a,
    sec_gsec11,
    sec_gsec12,
    sec_gsec13,
    sec_gsec14,
    sec_gsec15a,
    sec_gsec15b,
    sec_gsec15c,
    sec_gsec15d,
    sec_gsec15e,
    sec_gsec16,
    sec_gsec17,
    sec_gsec18,
    sec_gsec18a,
    sec_gsec18b,
    sec_unps
]

hhiddata = hh_sections[0]

for sec in hh_sections[1:]:
    hhiddata = hhiddata.merge(sec, on="HHID", how="left", validate="1:1")

print(hhiddata.shape)
print(hhiddata.head())

(2942, 120)
         HHID  H9Q07  H9Q08  H9Q09A  H9Q16  H9Q20  H9Q21  H9Q22  H9Q23  \
0  1013000201    3.0    NaN    10.0    9.0    3.0    2.0    5.0    1.0   
1  1013000204    6.0    6.0     5.0    2.0    2.0    5.0    2.0    1.0   
2  1013000206    2.0    NaN     3.0    9.0    2.0    5.0    4.0    1.0   
3  1013000210    3.0    NaN    10.0    1.0    1.0    5.0    2.0    1.0   
4  1013000213    3.0    NaN    15.0    1.0    2.0    5.0    8.0    1.0   

   H10Q13  ...   LAT_MOD    LON_MOD  DIST_ROAD  DIST_POPCENTER  DIST_MARKET  \
0     6.0  ... -0.530628  32.327492      40.48           65.55        65.55   
1     3.0  ... -0.530628  32.327492      40.48           65.55        65.55   
2     5.0  ...  0.289081  32.560650       0.81            4.08         4.08   
3     6.0  ... -0.530628  32.327492      41.36           66.64        66.64   
4     8.0  ... -0.530628  32.327492      41.40           66.69        66.69   

   FSRAD3_AGPCT  FSRAD3_LCMAJ  SSA_AEZ09  AFMNSLP_PCT  SRTM_UGA  
0 

In [8]:
# 2. Variable list PID
vars_gsec3_pid = [
    "HHID", "PID",
    "H3Q1", "H3Q3", "H3Q6"
]

vars_gsec4_pid = [
    "HHID", "PID",
    "H4Q4", "H4Q5", "H4Q6", "H4Q11", "H4Q13", "H4Q15G"
]

sec_gsec3_pid = prep_pid_section(gsec3, vars_gsec3_pid)
sec_gsec4_pid = prep_pid_section(gsec4, vars_gsec4_pid)

piddata = sec_gsec3_pid.merge(
    sec_gsec4_pid,
    on=["HHID", "PID"],
    how="left",
    validate="1:1"
)

print(piddata.shape)
print(piddata.head())

print("Unique HHID-PID pairs:", piddata[["HHID", "PID"]].drop_duplicates().shape[0])
print("Total rows:", len(piddata))

Duplicates on HHID-PID: 0
Duplicates on HHID-PID: 0
(16512, 11)
         HHID           PID  H3Q1  H3Q3  H3Q6  H4Q4  H4Q5  H4Q6  H4Q11  H4Q13  \
0  1013000201  101300020101     1   NaN   NaN   4.0   2.0   NaN    NaN    NaN   
1  1013000201  101300020102     2   NaN   NaN   4.0   2.0   NaN    NaN    NaN   
2  1013000201  101300020103     3   3.0   NaN   1.0   2.0   NaN    NaN    NaN   
3  1013000201  101300020104     4   2.0   NaN   4.0   3.0   NaN    2.0    1.6   
4  1013000201  101300020105     5   2.0   NaN   1.0   3.0   NaN    4.0    1.6   

     H4Q15G  
0       NaN  
1       NaN  
2       NaN  
3  460000.0  
4  270000.0  
Unique HHID-PID pairs: 16512
Total rows: 16512


In [9]:
import pandas as pd

roster = gsec2[["HHID", "PID", "H2Q3", "H2Q8", "H2Q5"]].copy()

roster.columns = roster.columns.str.strip().str.upper()

roster = roster.rename(columns={
    "H2Q3": "sex",
    "H2Q8": "age",
    "H2Q5": "months_present"
})

roster["HHID"] = roster["HHID"].astype(str).str.strip()
roster["PID"] = roster["PID"].astype(str).str.strip()

roster["age"] = pd.to_numeric(roster["age"], errors="coerce")
roster["months_present"] = pd.to_numeric(roster["months_present"], errors="coerce")

roster["adult"] = roster["age"] >= 18
roster["child"] = roster["age"] < 18

hh_roster = (
    roster.groupby("HHID", dropna=False)
    .agg(
        hh_size=("PID", "count"),
        adults=("adult", "sum"),
        children=("child", "sum"),
        avg_age=("age", "mean"),
        months_present_avg=("months_present", "mean"),
        months_present_sum=("months_present", "sum")
    )
    .reset_index()
)

roster["male"] = roster["sex"] == 1
roster["female"] = roster["sex"] == 2

hh_roster = (
    roster.groupby("HHID", dropna=False)
    .agg(
        hh_size=("PID", "count"),
        adults=("adult", "sum"),
        children=("child", "sum"),
        males=("male", "sum"),
        females=("female", "sum")
    )
    .reset_index()
)

In [11]:
from pathlib import Path

OUT = Path("/Output/interim")
OUT.mkdir(parents=True, exist_ok=True)

hhiddata["HHID"] = "'" + hhiddata["HHID"].astype(str)
piddata["HHID"] = "'" + piddata["HHID"].astype(str)
hh_roster["HHID"] = "'" + hh_roster["HHID"].astype(str)

hhiddata.to_csv(OUT / "hhiddata1_wave1.csv", index=False)
piddata.to_csv(OUT / "piddata_wave1.csv", index=False)
hh_roster.to_csv(OUT / "hh_roster_wave1.csv", index=False)



In [9]:
keep = {
    "gsec1", "gsec2", "gsec3", "gsec4",
    "gsec9", "gsec10", "gsec10a" "gsec11", "gsec12",
    "gsec13", "gsec14",
    "gsec15a", "gsec15b", "gsec15c", "gsec15d", "gsec15e",
    "gsec16", "gsec17", "gsec18", "gsec18a", "gsec18b",
    "geovars", "pov09",
    "piddata", "hhiddata"
}

for name in list(globals().keys()):
    if name not in keep and not name.startswith("_"):
        del globals()[name]

In [19]:
# sections community

csec1 = load_dta(ROOT / "CSEC1.dta")
csec2a = load_dta(ROOT / "CSEC2A.dta")
csec2b = load_dta(ROOT / "CSEC2B.dta")
csec2c = load_dta(ROOT / "CSEC2C.dta")
csec3a = load_dta(ROOT / "CSEC3A.dta")
csec3e = load_dta(ROOT / "CSEC3E.dta")
csec3f = load_dta(ROOT / "CSEC3F.dta")
csec3k = load_dta(ROOT / "CSEC3K.dta")
csec4b = load_dta(ROOT / "CSEC4B.dta")
csec4c = load_dta(ROOT / "CSEC4C.dta")
csec4d = load_dta(ROOT / "CSEC4D.dta")
csec4E = load_dta(ROOT / "CSEC4E.dta")
csec4f = load_dta(ROOT / "CSEC4F.dta")
csec4k3 = load_dta(ROOT / "CSEC4K3.dta")
csec4l = load_dta(ROOT / "CSEC4L.dta")
csec5a = load_dta(ROOT / "CSEC5A.dta")
csec5c = load_dta(ROOT / "CSEC5C.dta")
csec5d = load_dta(ROOT / "CSEC5D.dta")



In [21]:
#Community agg work

import pandas as pd

def keep_existing(df, cols):

    cols_upper = [c.upper() for c in cols]
    existing = [c for c in cols_upper if c in df.columns]
    return df[existing].copy()


def standardize_comcod(df):

    df = df.copy()
    df.columns = df.columns.str.strip().str.upper()

    if "COMCOD" not in df.columns:
        raise KeyError("COMCOD not found in dataframe")

    df["COMCOD"] = df["COMCOD"].astype(str).str.strip()
    return df


def prep_comm_section(df, keep_cols, section_name):

    df = df.copy()
    df.columns = df.columns.str.strip().str.upper()

    df = keep_existing(df, keep_cols)
    df = standardize_comcod(df)

    dup_count = df.duplicated(subset=["COMCOD"]).sum()
    print(f"{section_name}: duplicates on COMCOD = {dup_count}")

    if dup_count > 0:
        dup = df[df.duplicated(subset=["COMCOD"], keep=False)].sort_values("COMCOD")
        print(f"Warning: {section_name} has repeated COMCOD values. Inspect before forcing merge.")
        print(dup.head(20))

    return df


def agg_comm_to_comcod(df):

    df = df.copy()

    numeric_cols = [
        c for c in df.columns
        if c != "COMCOD" and pd.api.types.is_numeric_dtype(df[c])
    ]

    non_numeric_cols = [
        c for c in df.columns
        if c != "COMCOD" and c not in numeric_cols
    ]

    agg_map = {c: "mean" for c in numeric_cols}
    for c in non_numeric_cols:
        agg_map[c] = "first"

    out = df.groupby("COMCOD", dropna=False).agg(agg_map).reset_index()
    return out

vars_csec1 = [
    "YEAR", "COMCOD", "COMM", "CMULT", "C1AQ1", "C1AQ2", "C1AQ3", "C1AQ4", "C1AQ5"
]

vars_csec2a = [
    "COMCOD", "C2AQ2", "C2AQ3"
]

vars_csec2b = [
    "COMCOD", "C2BQ9", "C2BQ10", "C2BQ13", "C2BQ14"
]

vars_csec2c = [
    "COMCOD", "C2CQ15", "C2CQ16", "C2CQ17", "C2CQ18"
]

vars_csec3a = [
    "COMCOD", "C3AQ8", "C3AQ9", "C3AQ10"
]

vars_csec3e = [
    "COMCOD",
    "C3EQ30A", "C3EQ30B",
    "C3EQ31A", "C3EQ31B",
    "C3EQ32A", "C3EQ32B",
    "C3EQ33A", "C3EQ33B"
]

vars_csec3f = [
    "COMCOD", "C3F", "C3FQ34", "C3FQ35A", "C3FQ35B"
]

vars_csec3k = [
    "COMCOD", "C3LQ61", "C3LQ62"
]

vars_csec4b = [
    "COMCOD", "C4BQ16", "C4BQ17", "C4BQ22", "C4BQ25", "C4BQ28"
]

vars_csec4c = [
    "COMCOD", "C4CQ45", "C4CQ46", "C4CQ47"
]

vars_csec4d = [
    "COMCOD", "C4DQ49", "C4DQ50", "C4DQ51", "C4DQ52", "C4DQ53",
    "C4DQ54", "C4DQ55", "C4DQ56", "C4DQ57", "C4DQ58"
]

vars_csec4e = [
    "COMCOD", "C4EQ60", "C4EQ61", "C4EQ62", "C4EQ63"
]

vars_csec4f = [
    "COMCOD", "C4FQ64"
]

vars_csec4k3 = [
    "COMCOD", "C4KQ102", "C4KQ103", "C4KQ104", "C4KQ105"
]

vars_csec4l = [
    "COMCOD", "C4LQ106", "C4LQ107", "C4LQ108"
]

vars_csec5a = [
    "COMCOD", "C5AQ4", "C5AQ5", "C5AQ6", "C5AQ7", "C5AQ8", "C5AQ9"
]

vars_csec5c = [
    "COMCOD", "C5CSN", "C5C", "C5CQ18"
]

vars_csec5d = [
    "COMCOD", "C5DQ21", "C5DQ22", "C5DQ23"
]

sec_csec1 = prep_comm_section(csec1, vars_csec1, "CSEC1")
sec_csec2a = prep_comm_section(csec2a, vars_csec2a, "CSEC2A")
sec_csec2b = prep_comm_section(csec2b, vars_csec2b, "CSEC2B")
sec_csec2c = prep_comm_section(csec2c, vars_csec2c, "CSEC2C")
sec_csec3a = prep_comm_section(csec3a, vars_csec3a, "CSEC3A")
sec_csec3e = prep_comm_section(csec3e, vars_csec3e, "CSEC3E")
sec_csec3f = prep_comm_section(csec3f, vars_csec3f, "CSEC3F")
sec_csec3k = prep_comm_section(csec3k, vars_csec3k, "CSEC3K")
sec_csec4b = prep_comm_section(csec4b, vars_csec4b, "CSEC4B")
sec_csec4c = prep_comm_section(csec4c, vars_csec4c, "CSEC4C")
sec_csec4d = prep_comm_section(csec4d, vars_csec4d, "CSEC4D")
sec_csec4e = prep_comm_section(csec4E, vars_csec4e, "CSEC4E")
sec_csec4f = prep_comm_section(csec4f, vars_csec4f, "CSEC4F")
sec_csec4k3 = prep_comm_section(csec4k3, vars_csec4k3, "CSEC4K3")
sec_csec4l = prep_comm_section(csec4l, vars_csec4l, "CSEC4L")
sec_csec5a = prep_comm_section(csec5a, vars_csec5a, "CSEC5A")
sec_csec5c = prep_comm_section(csec5c, vars_csec5c, "CSEC5C")
sec_csec5d = prep_comm_section(csec5d, vars_csec5d, "CSEC5D")

comm_sections = {
    "CSEC1": sec_csec1,
    "CSEC2A": sec_csec2a,
    "CSEC2B": sec_csec2b,
    "CSEC2C": sec_csec2c,
    "CSEC3A": sec_csec3a,
    "CSEC3E": sec_csec3e,
    "CSEC3F": sec_csec3f,
    "CSEC3K": sec_csec3k,
    "CSEC4B": sec_csec4b,
    "CSEC4C": sec_csec4c,
    "CSEC4D": sec_csec4d,
    "CSEC4E": sec_csec4e,
    "CSEC4F": sec_csec4f,
    "CSEC4K3": sec_csec4k3,
    "CSEC4L": sec_csec4l,
    "CSEC5A": sec_csec5a,
    "CSEC5C": sec_csec5c,
    "CSEC5D": sec_csec5d
}

for name, df in comm_sections.items():
    print(name, len(df), df["COMCOD"].nunique())


def make_safe_name(x):
    x = str(x).strip().lower()
    for old, new in [
        ("/", "_"),
        ("-", "_"),
        ("(", ""),
        (")", ""),
        (",", ""),
        (".", ""),
        (" ", "_"),
        ("&", "and"),
        ("__", "_"),
    ]:
        x = x.replace(old, new)
    return x


def prep_comm(df, id_col="COMCOD"):
    df = df.copy()
    df.columns = df.columns.str.strip().str.upper()
    df[id_col] = df[id_col].astype(str).str.strip()
    return df


def pivot_comm_section(
    df,
    section_name,
    id_col,
    category_col,
    value_cols,
    aggfunc="first"
):

    work = prep_comm(df, id_col=id_col)

    keep_cols = [id_col, category_col] + value_cols
    keep_cols = [c.upper() for c in keep_cols]
    work = work[[c for c in keep_cols if c in work.columns]].copy()

    work[category_col.upper()] = work[category_col.upper()].apply(make_safe_name)

    out = None

    for val in [c.upper() for c in value_cols if c.upper() in work.columns]:
        wide = work.pivot_table(
            index=id_col.upper(),
            columns=category_col.upper(),
            values=val,
            aggfunc=aggfunc
        ).reset_index()

        wide.columns = [
            id_col.upper() if c == id_col.upper() else f"{section_name}__{val}__{c}"
            for c in wide.columns
        ]

        if out is None:
            out = wide
        else:
            out = out.merge(wide, on=id_col.upper(), how="outer", validate="1:1")

    if out is None:
        out = work[[id_col.upper()]].drop_duplicates()

    return out


def aggregate_comm_section(df, section_name, id_col="COMCOD", aggfunc="mean"):

    work = prep_comm(df, id_col=id_col)

    numeric_cols = [
        c for c in work.columns
        if c != id_col.upper() and pd.api.types.is_numeric_dtype(work[c])
    ]

    if not numeric_cols:
        return work[[id_col.upper()]].drop_duplicates()

    out = work.groupby(id_col.upper(), dropna=False)[numeric_cols].agg(aggfunc).reset_index()

    out = out.rename(columns={
        c: f"{section_name}__{c}" for c in out.columns if c != id_col.upper()
    })

    return out


def merge_comm(base, other, id_col="COMCOD"):
    if base is None or base.empty:
        return other.copy()
    return base.merge(other, on=id_col.upper(), how="left", validate="1:1")


community_config = {
    "CSEC1": {
        "type": "aggregate"
    },

    # repeated sections with category labels -> pivot wide
    "CSEC2A": {
        "type": "pivot",
        "category_col": "C2AQ2",
        "value_cols": ["C2AQ3"]
    },
    "CSEC2B": {
        "type": "pivot",
        "category_col": "C2BQ9",
        "value_cols": ["C2BQ10", "C2BQ13", "C2BQ14"]
    },
    "CSEC2C": {
        "type": "pivot",
        "category_col": "C2CQ15",
        "value_cols": ["C2CQ16", "C2CQ17", "C2CQ18"]
    },
    "CSEC3A": {
        "type": "aggregate"
    },
    "CSEC3E": {
        "type": "aggregate"
    },
    "CSEC3F": {
        "type": "pivot",
        "category_col": "C3F",
        "value_cols": ["C3FQ34", "C3FQ35A", "C3FQ35B"]
    },
    "CSEC3K": {
        "type": "aggregate"
    },
    "CSEC4B": {
        "type": "aggregate"
    },
    "CSEC4C": {
        "type": "aggregate"
    },
    "CSEC4D": {
        "type": "aggregate"
    },
    "CSEC4E": {
        "type": "aggregate"
    },
    "CSEC4F": {
        "type": "aggregate"
    },
    "CSEC4K3": {
        "type": "aggregate"
    },
    "CSEC4L": {
        "type": "aggregate"
    },
    "CSEC5A": {
        "type": "aggregate"
    },
    "CSEC5C": {
        "type": "pivot",
        "category_col": "C5CSN",
        "value_cols": ["C5C", "C5CQ18"]
    },
    "CSEC5D": {
    "type": "pivot",
    "category_col": "C5DQ21",
    "value_cols": ["C5DQ22", "C5DQ23"]
    }
}

community_raw = {
    "CSEC1": csec1,
    "CSEC2A": csec2a,
    "CSEC2B": csec2b,
    "CSEC2C": csec2c,
    "CSEC3A": csec3a,
    "CSEC3E": csec3e,
    "CSEC3F": csec3f,
    "CSEC3K": csec3k,
    "CSEC4B": csec4b,
    "CSEC4C": csec4c,
    "CSEC4D": csec4d,
    "CSEC4E": csec4E,
    "CSEC4F": csec4f,
    "CSEC4K3": csec4k3,
    "CSEC4L": csec4l,
    "CSEC5A": csec5a,
    "CSEC5C": csec5c,
    "CSEC5D": csec5d,
}

community_clean = {}

for sec_name, df in community_raw.items():
    cfg = community_config[sec_name]

    if cfg["type"] == "pivot":
        clean_df = pivot_comm_section(
            df=df,
            section_name=sec_name,
            id_col="COMCOD",
            category_col=cfg["category_col"],
            value_cols=cfg["value_cols"],
            aggfunc="first"
        )
    else:
        clean_df = aggregate_comm_section(
            df=df,
            section_name=sec_name,
            id_col="COMCOD",
            aggfunc="mean"
        )

    community_clean[sec_name] = clean_df
    print(sec_name, clean_df.shape)

commdata = None

for sec_name in community_clean:
    commdata = merge_comm(commdata, community_clean[sec_name], id_col="COMCOD")

print(commdata.shape)
commdata.head()

CSEC1: duplicates on COMCOD = 0
CSEC2A: duplicates on COMCOD = 7766
     COMCOD                 C2AQ2  C2AQ3
0   1010002  Government Primary S    2.0
26  1010002  Army detach/barracks    3.0
25  1010002  Police station or Po    3.0
24  1010002   Fisheries extension    1.0
23  1010002  Agricultural extensi    1.0
22  1010002   Veterinary Services    3.0
21  1010002    Traditional healer    3.0
20  1010002        Community road    3.0
18  1010002   Trunk road (murram)    3.0
17  1010002   Trunk road (tarmac)    3.0
16  1010002  Market selling non-a    3.0
15  1010002  Market selling agric    1.0
14  1010002  Market selling agric    3.0
19  1010002  Feeder/ District roa    3.0
12  1010002  Bank/financial insti    3.0
13  1010002           Post office    3.0
2   1010002  Government Secondary    3.0
3   1010002  Private Secondary Sc    3.0
4   1010002  Technical/Vocational    3.0
5   1010002  Alternative Basic Ed    3.0
CSEC2B: duplicates on COMCOD = 0
CSEC2C: duplicates on COMCOD = 0
CSEC3

In [2]:
# sections agriculture

agsec1 = load_dta(ROOT / "AGSEC1.dta")
agsec10 = load_dta(ROOT / "AGSEC10.dta")
agsec2 = load_dta(ROOT / "AGSEC2.dta")
agsec2a = load_dta(ROOT / "AGSEC2A.dta")
agsec2b = load_dta(ROOT / "AGSEC2B.dta")
agsec3a = load_dta(ROOT / "AGSEC3A.dta")
agsec4a = load_dta(ROOT / "AGSEC4A.dta")
agsec5a = load_dta(ROOT / "AGSEC5A.dta")
agsec6a = load_dta(ROOT / "AGSEC6A.dta")
agsec6b = load_dta(ROOT / "AGSEC6B.dta")
agsec6c = load_dta(ROOT / "AGSEC6C.dta")



In [ ]:
#Agriculture section logic
"""
AGSEC1   household-level base
AGSEC2   household-level base / summary
AGSEC2A  parcel roster, key = HHID + parcel_id
AGSEC2B  parcel roster, key = HHID + parcel_id
AGSEC3A  parcel-level / plot-level, key = HHID + parcel_id + possible multiple plot id for each parcel id (Q3)
AGSEC4A  parcel-season / crop module, key = HHID + parcel_id + possible multiple plot id for each parcel id (Q4) + multiple crop code (Q6)
AGSEC5A  parcel-season / crop module, key = HHID + parcel_id + possible multiple plot id for each parcel id (Q3) + multiple crop code (Q5)
AGSEC6A  livestock module, key = HHID + livestock index (Large animals)
AGSEC6B  livestock module, key = HHID + livestock index (small animals)
AGSEC6C  livestock module, key = HHID + livestock index (poultry)
AGSEC10  service/contact module, key = HHID + service_id
"""


In [3]:
#Helper func

import pandas as pd


def keep_existing(df, cols):
    cols_upper = [c.upper() for c in cols]
    existing = [c for c in cols_upper if c in df.columns]
    return df[existing].copy()


def standardize_hhid(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.upper()

    if "HHID" not in df.columns:
        raise KeyError("HHID not found in dataframe")

    df["HHID"] = df["HHID"].astype(str).str.strip()
    return df


def prep_ag_section(df, keep_cols, section_name):

    df = df.copy()
    df.columns = df.columns.str.strip().str.upper()

    df = keep_existing(df, keep_cols)
    df = standardize_hhid(df)

    print(f"\n{section_name}")
    print("shape:", df.shape)
    print("unique HHID:", df["HHID"].nunique())
    print("rows per HHID, top 10:")
    print(df["HHID"].value_counts().head(10))

    return df


def check_possible_keys(df, section_name, candidates=None):

    if candidates is None:
        candidates = [
            "PID", "PARCELID", "PLOTID", "PLOT", "FIELD",
            "CROPID", "CROP", "LIVESTOCKID", "ENTERPRISEID",
            "A2AQ2", "A2BQ2", "A3AQ1", "A4AQ2", "A5AQ1", "A5BQ1", "A6CQ2", "A6BQ2", "A6AQ2"
        ]

    cols = [c for c in candidates if c in df.columns]
    print(f"\n{section_name} possible keys present:", cols)

    for c in cols:
        n = df[[ "HHID", c ]].drop_duplicates().shape[0]
        print(f"unique HHID + {c}: {n}")

In [4]:
 #variable list

vars_agsec1 = [
    "YEAR", "HHID", "A2AQ1", "A2BQ1", "A4AQ1", "A4BQ1", "A6AQ1",
    "A6AQ21", "A6AQ22", "A6BQ1", "A6CQ1", "A8", "A8Q11", "A8Q12",
    "A10Q11", "A10Q12", "A10Q13", "A10Q14", "A10Q15", "A19Q16"
]

vars_agsec10 = [
    "HHID", "A10Q2", "A10Q3", "A10Q5A", "A10Q5B", "A10Q5C",
    "A10Q5D", "A10Q8", "A10Q9", "A10Q10"
]

vars_agsec2 = [
    "HHID", "A10Q11", "A10Q12", "A10Q13", "A10Q14", "A10Q15", "A19Q16"
]

vars_agsec2a = [
    "HHID", "A2AQ2", "A2AQ4", "A2AQ6", "A2AQ7", "A2AQ13A", "A2AQ13B", "A2AQ17A",
    "A2AQ17B", "A2AQ18", "A2AQ19", "A2AQ20", "A2AQ21"
]

vars_agsec2b = [
    "HHID", "A2BQ2", "A2BQ4", "A2BQ15A", "A2BQ15B", "A2BQ17",
    "A2BQ18", "A2BQ19", "A2BQ20"
]

vars_agsec3a = [
    "HHID", "A3AQ1", "A3AQ3", "A3AQ4", "A3AQ5", "A3AQ6", "A3AQ7",
    "A3AQ8", "A3AQ9", "A3AQ10", "A3AQ13", "A3AQ14", "A3AQ15",
    "A3AQ16", "A3AQ17", "A3AQ18", "A3AQ19", "A3AQ21", "A3AQ22",
    "A3AQ25", "A3AQ26", "A3AQ27", "A3AQ28A", "A3AQ28B", "A3AQ29",
    "A3AQ30", "A3AQ31", "A3AQ33", "A3AQ34", "A3AQ37", "A3AQ38"
]

vars_agsec4a = [
    "HHID", "A4AQ1", "A4AQ2", "A4AQ4", "A4AQ5", "A4AQ6", "A4AQ7", "A4AQ8",
    "A4AQ9", "A4AQ10", "A4AQ11", "A4AQ12", "A4AQ13", "A4AQ14"
]

vars_agsec5a = [
    "HHID", "A5AQ1", "A5AQ3", "A5AQ4", "A5AQ5", "A5AQ6A", "A5AQ6B",
    "A5AQ6C", "A5AQ6D", "A5AQ7A", "A5AQ7B", "A5AQ7C", "A5AQ8",
    "A5AQ9", "A5AQ15", "A5AQ16", "A5AQ17", "A5AQ18", "A5AQ19"
]

vars_agsec6a = [
    "HHID","A6AQ2", "A6AQ3", "A6AQ4", "A6AQ5", "A6AQ18"
]

vars_agsec6b = [
    "HHID", "A6BQ2", "A6BQ3", "A6BQ4", "A6BQ5"
]

vars_agsec6c = [
    "HHID", "A6CQ2", "A6CQ3", "A6CQ4", "A6CQ5"
]

#prep

sec_agsec1 = prep_ag_section(agsec1, vars_agsec1, "AGSEC1")
sec_agsec10 = prep_ag_section(agsec10, vars_agsec10, "AGSEC10")
sec_agsec2 = prep_ag_section(agsec2, vars_agsec2, "AGSEC2")
sec_agsec2a = prep_ag_section(agsec2a, vars_agsec2a, "AGSEC2A")
sec_agsec2b = prep_ag_section(agsec2b, vars_agsec2b, "AGSEC2B")
sec_agsec3a = prep_ag_section(agsec3a, vars_agsec3a, "AGSEC3A")
sec_agsec4a = prep_ag_section(agsec4a, vars_agsec4a, "AGSEC4A")
sec_agsec5a = prep_ag_section(agsec5a, vars_agsec5a, "AGSEC5A")
sec_agsec6a = prep_ag_section(agsec6a, vars_agsec6a, "AGSEC6A")
sec_agsec6b = prep_ag_section(agsec6b, vars_agsec6b, "AGSEC6B")
sec_agsec6c = prep_ag_section(agsec6c, vars_agsec6c, "AGSEC6C")

check_possible_keys(sec_agsec2a, "AGSEC2A")
check_possible_keys(sec_agsec2b, "AGSEC2B")
check_possible_keys(sec_agsec3a, "AGSEC3A")
check_possible_keys(sec_agsec4a, "AGSEC4A")
check_possible_keys(sec_agsec5a, "AGSEC5A")
check_possible_keys(sec_agsec6a, "AGSEC6A")
check_possible_keys(sec_agsec6b, "AGSEC6B")
check_possible_keys(sec_agsec6c, "AGSEC6C")
check_possible_keys(sec_agsec10, "AGSEC10")


AGSEC1
shape: (2428, 20)
unique HHID: 2428
rows per HHID, top 10:
HHID
1013000201      1
1013000204      1
1013000210      1
101300021302    1
1021000108      1
1021000111      1
1021000113      1
1021000408      1
1021000710      1
1021000807      1
Name: count, dtype: int64

AGSEC10
shape: (11970, 10)
unique HHID: 2004
rows per HHID, top 10:
HHID
                7
211100030705    6
304100041003    6
317300091006    6
304300091102    6
302300131005    6
107100010608    6
408300040304    6
212300080902    6
111300030203    6
Name: count, dtype: int64

AGSEC2
shape: (2428, 7)
unique HHID: 2428
rows per HHID, top 10:
HHID
1013000201      1
1013000204      1
1013000210      1
101300021302    1
1021000108      1
1021000111      1
1021000113      1
1021000408      1
1021000710      1
1021000807      1
Name: count, dtype: int64

AGSEC2A
shape: (4305, 13)
unique HHID: 2133
rows per HHID, top 10:
HHID
3103000807      13
3023000310      11
3043000902      10
3103000811      10
310300100303    

In [5]:
def clean_id(x):
    if pd.isna(x):
        return pd.NA
    try:
        return str(int(float(x)))
    except Exception:
        return str(x).strip()


def standardize_hhid(df):
    df = df.copy()
    df["HHID"] = df["HHID"].astype(str).str.strip()
    return df


# 2. Household-level agriculture base

ag_hhid = sec_agsec1.merge(
    sec_agsec2,
    on="HHID",
    how="left",
    validate="1:1",
    suffixes=("", "_AGSEC2")
)


print("ag_hhid:", ag_hhid.shape)



# 3. Parcel roster: owned + user-rights parcels

owned_parcels = sec_agsec2a.copy()
owned_parcels["parcel_id"] = owned_parcels["A2AQ2"].apply(clean_id)
owned_parcels["parcel_type"] = "owned"

rights_parcels = sec_agsec2b.copy()
rights_parcels["parcel_id"] = rights_parcels["A2BQ2"].apply(clean_id)
rights_parcels["parcel_type"] = "user_rights"

parcel_roster = pd.concat(
    [owned_parcels, rights_parcels],
    ignore_index=True
)

parcel_roster = standardize_hhid(parcel_roster)
parcel_roster = parcel_roster.drop_duplicates(subset=["HHID", "parcel_id", "parcel_type"])

print("parcel_roster:", parcel_roster.shape)
print("unique HHID-parcel:", parcel_roster[["HHID", "parcel_id"]].drop_duplicates().shape[0])



# 4. Plot-level section: AGSEC3A

ag3a_plot = sec_agsec3a.copy()
ag3a_plot = standardize_hhid(ag3a_plot)

ag3a_plot["parcel_id"] = ag3a_plot["A3AQ1"].apply(clean_id)
ag3a_plot["plot_id"] = ag3a_plot["A3AQ3"].apply(clean_id)

print("ag3a_plot:", ag3a_plot.shape)
print("duplicates plot key:",
      ag3a_plot.duplicated(["HHID", "parcel_id", "plot_id"]).sum())



# 5. Crop-level section: AGSEC4A
# parcel = A4AQ2, plot = A4AQ4, crop = A4AQ6

ag4a_crop = sec_agsec4a.copy()
ag4a_crop = standardize_hhid(ag4a_crop)

ag4a_crop["parcel_id"] = ag4a_crop["A4AQ2"].apply(clean_id)
ag4a_crop["plot_id"] = ag4a_crop["A4AQ4"].apply(clean_id)
ag4a_crop["crop_id"] = ag4a_crop["A4AQ6"].apply(clean_id)

print("ag4a_crop:", ag4a_crop.shape)
print("duplicates crop key:",
      ag4a_crop.duplicated(["HHID", "parcel_id", "plot_id", "crop_id"]).sum())



# 6. Crop-level section: AGSEC5A
# parcel = A5AQ1, plot = A5AQ3, crop = A5AQ5

ag5a_crop = sec_agsec5a.copy()
ag5a_crop = standardize_hhid(ag5a_crop)

ag5a_crop["parcel_id"] = ag5a_crop["A5AQ1"].apply(clean_id)
ag5a_crop["plot_id"] = ag5a_crop["A5AQ3"].apply(clean_id)
ag5a_crop["crop_id"] = ag5a_crop["A5AQ5"].apply(clean_id)

print("ag5a_crop:", ag5a_crop.shape)
print("duplicates crop key:",
      ag5a_crop.duplicated(["HHID", "parcel_id", "plot_id", "crop_id"]).sum())



# 7. Build plot roster from AGSEC3A, AGSEC4A, AGSEC5A

plot_roster = pd.concat(
    [
        ag3a_plot[["HHID", "parcel_id", "plot_id"]],
        ag4a_crop[["HHID", "parcel_id", "plot_id"]],
        ag5a_crop[["HHID", "parcel_id", "plot_id"]],
    ],
    ignore_index=True
).drop_duplicates()

print("plot_roster:", plot_roster.shape)


ag4a_plot_agg = (
    ag4a_crop
    .groupby(["HHID", "parcel_id", "plot_id"], dropna=False)
    .agg(
        n_crops_ag4a=("crop_id", "nunique")
    )
    .reset_index()
)

ag5a_plot_agg = (
    ag5a_crop
    .groupby(["HHID", "parcel_id", "plot_id"], dropna=False)
    .agg(
        n_crops_ag5a=("crop_id", "nunique")
    )
    .reset_index()
)



# 9. Plot-level dataset

# Remove original key columns before merge to avoid confusion
ag3a_plot_merge = ag3a_plot.drop(columns=["A3AQ1", "A3AQ3"], errors="ignore")

plotdata = plot_roster.copy()

plotdata = plotdata.merge(
    ag3a_plot_merge,
    on=["HHID", "parcel_id", "plot_id"],
    how="left",
    validate="1:1"
)

plotdata = plotdata.merge(
    ag4a_plot_agg,
    on=["HHID", "parcel_id", "plot_id"],
    how="left",
    validate="1:1"
)

plotdata = plotdata.merge(
    ag5a_plot_agg,
    on=["HHID", "parcel_id", "plot_id"],
    how="left",
    validate="1:1"
)

print("plotdata:", plotdata.shape)


ag6a_livestock = sec_agsec6a.copy()
ag6a_livestock = standardize_hhid(ag6a_livestock)
ag6a_livestock["livestock_id"] = ag6a_livestock["A6AQ2"].apply(clean_id)
ag6a_livestock["livestock_group"] = "large_animals"

ag6b_livestock = sec_agsec6b.copy()
ag6b_livestock = standardize_hhid(ag6b_livestock)
ag6b_livestock["livestock_id"] = ag6b_livestock["A6BQ2"].apply(clean_id)
ag6b_livestock["livestock_group"] = "small_animals"

ag6c_livestock = sec_agsec6c.copy()
ag6c_livestock = standardize_hhid(ag6c_livestock)
ag6c_livestock["livestock_id"] = ag6c_livestock["A6CQ2"].apply(clean_id)
ag6c_livestock["livestock_group"] = "poultry"

livestockdata = pd.concat(
    [ag6a_livestock, ag6b_livestock, ag6c_livestock],
    ignore_index=True
)

print("livestockdata:", livestockdata.shape)



# 11. Extension/service module

servicedata = sec_agsec10.copy()
servicedata = standardize_hhid(servicedata)
servicedata["service_id"] = servicedata["A10Q2"].apply(clean_id)

print("servicedata:", servicedata.shape)
print("duplicates service key:",
      servicedata.duplicated(["HHID", "service_id"]).sum())


ag_hhid: (2428, 26)
parcel_roster: (5825, 23)
unique HHID-parcel: 5825
ag3a_plot: (9898, 33)
duplicates plot key: 0
ag4a_crop: (13987, 17)
duplicates crop key: 0
ag5a_crop: (15403, 22)
duplicates crop key: 1416
plot_roster: (9898, 3)
plotdata: (9898, 33)
livestockdata: (39199, 16)
servicedata: (11970, 11)
duplicates service key: 1


In [7]:
from pathlib import Path

OUT = Path("/Output/interim")
OUT.mkdir(parents=True, exist_ok=True)

ag_hhid.to_csv(OUT / "ag_hhid_wave1.csv", index=False)
parcel_roster.to_csv(OUT / "ag_parcel_roster_wave1.csv", index=False)
plotdata.to_csv(OUT / "ag_plotdata_wave1.csv", index=False)

ag4a_crop.to_csv(OUT / "ag4a_crop_wave1.csv", index=False)
ag5a_crop.to_csv(OUT / "ag5a_crop_wave1.csv", index=False)

livestockdata.to_csv(OUT / "ag_livestock_wave1.csv", index=False)
servicedata.to_csv(OUT / "ag_services_wave1.csv", index=False)

In [11]:
def livestock_total(df, id_col, q5_col, new_name):
    work = df.copy()
    work["HHID"] = work["HHID"].astype(str).str.strip()
    work[q5_col] = pd.to_numeric(work[q5_col], errors="coerce")

    out = (
        work.groupby("HHID", dropna=False)[q5_col]
        .sum()
        .reset_index()
        .rename(columns={q5_col: new_name})
    )

    return out


large = livestock_total(sec_agsec6a, id_col="A6AQ2", q5_col="A6AQ5", new_name="total_large_owned")
small = livestock_total(sec_agsec6b, id_col="A6BQ2", q5_col="A6BQ5", new_name="total_small_owned")
poultry = livestock_total(sec_agsec6c, id_col="A6CQ2", q5_col="A6CQ5", new_name="total_poultry_owned")

livestockdata = (
    large
    .merge(small, on="HHID", how="outer", validate="1:1")
    .merge(poultry, on="HHID", how="outer", validate="1:1")
)

livestockdata.to_csv(OUT / "ag_livestock_wave1.csv", index=False)

print(livestockdata.shape)
livestockdata.head()

(1983, 4)


,HHID,total_large_owned,total_small_owned,total_poultry_owned
0,1013000201,NaN,2.0,10.0
1,1013000204,NaN,1.0,10.0
2,1013000210,NaN,10.0,NaN
3,101300021302,NaN,1.0,NaN
4,1021000111,NaN,NaN,15.0
